In [1]:
import globals as gl

import os
import numpy as np
import sys

import pandas as pd

import seaborn as sns
import matplotlib.pyplot as plt

sys.path.append('/Users/mnlmrc/Documents/GitHub')
sys.path.append('/Users/mnlmrc/Documents/GitHub/Functional_Fusion')
sys.path.append('/home/ROBARTS/memanue5/Documents/GitHub')
sys.path.append('/home/ROBARTS/memanue5/Documents/GitHub/Functional_Fusion')

from Functional_Fusion import reliability_between_subj

Base directory found: /Volumes/diedrichsen_data$/data/SensoriMotorPrediction/
Base directory: /Volumes/diedrichsen_data$/data/SensoriMotorPrediction/


In [ ]:
# Define parameters
experiment = 'smp2'
glm = 12
sn = 104

# Storage for SNR values
snr_dict = {
    'ROI': [],
    'Hemisphere': [],
    'SNR': []
}

# Iterate through hemispheres and ROIs
for Hem in ['L', 'R']:
    for roi in gl.rois['ROI']:
        # Load data
        Y = np.load(os.path.join(gl.baseDir, experiment, f'{gl.glmDir}{glm}', f'subj{sn}', f'ROI.{Hem}.{roi}.beta.npy'))
        res = np.load(os.path.join(gl.baseDir, experiment, f'{gl.glmDir}{glm}', f'subj{sn}', f'ROI.{Hem}.{roi}.res.npy'))

        # Prewhitening
        Y = Y / np.sqrt(res)
        
        Y = Y.reshape(1, Y.shape[0], Y.shape[1])

        # Load partitioning and condition information
        reginfo = pd.read_csv(os.path.join(gl.baseDir, experiment, f'{gl.glmDir}{glm}', f'subj100', f'subj100_reginfo.tsv'), sep='\t')
        part_vec = reginfo.run
        cond_vec = reginfo.name.str.strip()
        
        snr = reliability_within_subj(Y[:, cond_vec.isin(gl.cue)], part_vec[cond_vec.isin(gl.cue)], cond_vec[cond_vec.isin(gl.cue)],
                                              voxel_wise=False,
                                              subtract_mean=True)
        # snr_avg = snr.mean()
        # snr_err = snr.std()

        # Append to data list
        for s in snr[0]:
            snr_dict['ROI'].append(roi)
            snr_dict['Hemisphere'].append(Hem)
            snr_dict['SNR'].append(s)

# Convert to DataFrame
snr_df = pd.DataFrame(snr_dict)

# Pivot Data for Faceted Plot
fig, axs = plt.subplots()
sns.boxplot(data=snr_df, y='SNR', x='ROI', hue='Hemisphere', palette={'L': 'blue', 'R': 'red'}, dodge=True, ax=axs, showfliers=False)

# Formatting
# axs.set_xlabel("region")
axs.axhline(0, color='k', lw=.8)
axs.set_ylabel("SNR")
axs.set_title(f'SNR - glm{glm}, subj{sn}')
axs.legend(title="Hemisphere")
plt.show()